# Biểu diễn văn bản bằng túi từ (Bag of Words)

Mỗi câu được biểu diễn bằng một vector, mỗi phần tử là số lần xuất hiện của một từ trong từ điển. Dữ liệu viết thường, không có dấu câu để dễ tách bằng `split()`.

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

# Tập văn bản: mỗi phần tử là một câu
texts = [
    'tôi thích học máy',
    'tôi thích python',
    'học máy học python',
]

## Cách 1: Python thuần

Tách theo khoảng trắng (tiếng Việt khi đó được tách thành các âm tiết).

In [2]:
sentences = [text.split() for text in texts]          # mỗi câu thành danh sách từ
vocabulary = sorted(set(' '.join(texts).split()))     # từ điển: các từ khác nhau, sắp xếp

matrix = []                                           # ma trận câu x từ
for sentence in sentences:
    row = []
    for word in vocabulary:
        row.append(sentence.count(word))              # số lần từ xuất hiện trong câu
    matrix.append(row)

print('Từ điển:', vocabulary)
for row in matrix:
    print(row)

Từ điển: ['học', 'máy', 'python', 'thích', 'tôi']
[1, 1, 0, 1, 1]
[0, 0, 1, 1, 1]
[2, 1, 1, 0, 0]


## Cách 2: scikit-learn

Mỗi hàng là một câu, mỗi cột là số lần xuất hiện của một từ.

In [3]:
vectorizer = CountVectorizer()                        # bộ đếm từ
count_matrix = vectorizer.fit_transform(texts).toarray()

print('Từ điển:', vectorizer.get_feature_names_out().tolist())
print(count_matrix)

Từ điển: ['học', 'máy', 'python', 'thích', 'tôi']
[[1 1 0 1 1]
 [0 0 1 1 1]
 [2 1 1 0 0]]


## Cách 3: TF-IDF

Trọng số từ thay cho số lần xuất hiện: từ có trong ít câu được tăng trọng số; mỗi vector được chuẩn hóa L2.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(texts).toarray()   # ma trận trọng số TF-IDF

print('Từ điển:', tfidf.get_feature_names_out().tolist())
print(tfidf_matrix.round(3))

Từ điển: ['học', 'máy', 'python', 'thích', 'tôi']
[[0.5   0.5   0.    0.5   0.5  ]
 [0.    0.    0.577 0.577 0.577]
 [0.816 0.408 0.408 0.    0.   ]]


## Cách 4: Hashing

Băm từ vào vector cố định 8 chiều. Có thể có nhiều từ chung một cột; đây là biểu diễn bằng băm, không phải embedding học ngữ nghĩa.

In [5]:
from sklearn.feature_extraction.text import HashingVectorizer

hashing = HashingVectorizer(n_features=8, alternate_sign=False)   # vector 8 chiều
hash_matrix = hashing.transform(texts).toarray()
print(hash_matrix.round(3))

[[0.    0.816 0.    0.    0.    0.408 0.408 0.   ]
 [0.    0.577 0.    0.    0.    0.577 0.577 0.   ]
 [0.    0.707 0.    0.    0.    0.707 0.    0.   ]]


## Cách 5: LSA

Dùng SVD nén TF-IDF thành embedding 2 chiều.

In [6]:
from sklearn.decomposition import TruncatedSVD

lsa = TruncatedSVD(n_components=2, random_state=42)   # giữ 2 chiều ẩn
lsa_embedding = lsa.fit_transform(tfidf_matrix)
print(lsa_embedding.round(3))

[[ 0.921  0.017]
 [ 0.735 -0.639]
 [ 0.762  0.596]]


## Cách 6: NMF

Biểu diễn mỗi câu bằng 2 trọng số chủ đề không âm.

In [7]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=2, init='nndsvda', random_state=42, max_iter=500)   # 2 chủ đề
nmf_embedding = nmf.fit_transform(tfidf_matrix)
print(nmf_embedding.round(3))

[[0.433 0.482]
 [0.    0.83 ]
 [0.674 0.   ]]


Mỗi hàng vẫn là một câu; cột LSA/NMF là đặc trưng ẩn, không phải từ. Chỉ có 3 câu nên đây là minh họa cách chạy, chưa học ngữ nghĩa tốt.